# RQ3 — Auditabilità e riproducibilità
### *Possiamo fidarci della pipeline, e quanto costa usarla?*

> **Domanda di ricerca.** La pipeline è **auditabile** (ogni run lascia una traccia verificabile) e
> **riproducibile** (a parità di configurazione e seed produce lo stesso dataset sintetico)? E qual è il
> **costo computazionale** dei due metodi, per giustificare il trade-off costo/beneficio?

**Concetti usati in questo notebook**

| Concetto | Come lo verifichiamo | Colonna |
|---|---|---|
| **Auditabilità** | Ogni run registra *dove* sta (`run_dir`), *quando* è stata eseguita, *con quale configurazione* e *cosa ha prodotto* | `run_dir`, `timestamp`, `config_sha256`, `synthetic_sha256` |
| **Riproducibilità** | Ripetizioni della stessa run a **seed fisso** producono lo **stesso hash SHA-256** del dataset sintetico | `synthetic_sha256` |
| **Costo computazionale** | Tempo di addestramento e di campionamento del sintetizzatore, e quota sul tempo totale della run | `fit_seconds`, `sample_seconds`, `elapsed_seconds` |

Le ripetizioni di riproducibilità sono definite in `repeats_for_reproducibility` (`experiment_grid.yaml`).

---

### Guida alla lettura

| # | Sezione | Domanda a cui risponde |
|---|---------|------------------------|
| 1 | Setup | Configurazione, stile grafico, caricamento dati |
| 2 | I dati | Quante run ho, e per quali metodi posso verificare la riproducibilità? |
| 3 | Auditabilità | La traccia di ogni run è completa e coerente? |
| 4 | Riproducibilità | Le ripetizioni a seed fisso producono lo stesso output? |
| 5 | Costo computazionale | Quanto tempo richiedono i metodi, e dove va il tempo? |
| 6 | Costo vs beneficio | Il tempo in più di CTGAN si ripaga in utility? |
| 7 | Tabella di sintesi | Tutti i numeri in un colpo d'occhio (+ export LaTeX) |
| 8 | Limiti e prossimi passi | Cosa si può (e non si può) concludere |

> 🔄 **Questo notebook si adatta ai dati.** Non ci sono commenti scritti a mano sui risultati: le frasi sotto i
> grafici sono generate dal codice a partire dai dati caricati, quindi restano corrette quando rigeneri i run.
> Se non ci sono ripetizioni o mancano dei file, le sezioni interessate lo dicono invece di mostrare grafici vuoti.

---
## 1 · Setup
Stessa configurazione grafica dei notebook RQ1 e RQ2.

In [ ]:
import sys, json, ast
from pathlib import Path

# Root del progetto = la prima cartella (da qui in su) che contiene src/. Funziona sia che il kernel
# parta da notebooks/ sia dalla root, così i percorsi non dipendono dalla working directory.
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src").is_dir()), Path.cwd())
sys.path.insert(0, str(ROOT))     # per importare src.*

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import FuncFormatter, NullFormatter, PercentFormatter
from IPython.display import display, Markdown

from src.analysis.aggregate import load_runs_index

# ── Opzioni ────────────────────────────────────────────────────────────────
SAVE_FIGS   = False                                        # True → salva ogni figura in FIG_DIR (png 300 dpi + pdf)
FIG_DIR     = ROOT / "reports" / "figures" / "rq3"
INDEX_PATH  = ROOT / "reports" / "runs_index.jsonl"
BASE_METHOD = "gaussian_copula"                            # metodo di riferimento per il confronto costo/beneficio
PHASE_REPORT = "phase1_quality_report.json"                # file per-run con la chiave "phase_timings"

# ── Etichette leggibili e palette (colorblind-friendly) ────────────────────
DATASET_LABELS = {"adult_income": "Adult Income", "breast_cancer": "Breast Cancer", "german_credit": "German Credit"}
METHOD_LABELS  = {"gaussian_copula": "Gaussian Copula", "ctgan": "CTGAN"}
METHOD_COLORS  = {"gaussian_copula": "#2A9D8F", "ctgan": "#E76F51"}
C_TEXT, C_MUTED, C_BASE = "#1F2937", "#6B7280", "#9AA5B1"
C_OK, C_FAIL = "#2F9E62", "#D64545"                        # esito dei controlli (verde / rosso)
_FALLBACK = ["#457B9D", "#E9C46A", "#8E6C8A", "#6D9F71"]   # per metodi non previsti sopra

def label_of(method):  return METHOD_LABELS.get(method, method)
def color_of(method):
    if method in METHOD_COLORS: return METHOD_COLORS[method]
    return _FALLBACK[sorted(set(METHODS) - set(METHOD_COLORS)).index(method) % len(_FALLBACK)]

# ── Stile matplotlib ───────────────────────────────────────────────────────
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "figure.facecolor": "white",
    "font.size": 10.5, "text.color": C_TEXT, "axes.labelcolor": C_TEXT,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#CBD2D9", "xtick.color": C_MUTED, "ytick.color": C_MUTED,
    "axes.grid": True, "grid.color": "#E8ECEF", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False,
})

def finish(fig, name):
    # Chiude la figura: la salva (se SAVE_FIGS) e la mostra.
    if SAVE_FIGS:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        for ext in ("png", "pdf"):
            fig.savefig(FIG_DIR / f"{name}.{ext}")
    plt.show()

def fmt_int(n):  return f"{int(n):,}".replace(",", ".")
def fmt_pp(v):   return f"{v + np.sign(v) * 1e-9:+.1f} pp".replace("-", "−")
def fmt_time(s):
    # Secondi → testo leggibile (14 s, 12.4 min, 1.2 h).
    if pd.isna(s): return "n/d"
    if s < 10:     return f"{s:.1f} s"
    if s < 60:     return f"{s:.0f} s"
    if s < 3600:   return f"{s / 60:.1f} min"
    return f"{s / 3600:.1f} h"

In [ ]:
# ── Caricamento e preparazione ─────────────────────────────────────────────
df = load_runs_index(INDEX_PATH)

for c in ["fit_seconds", "sample_seconds", "elapsed_seconds", "accuracy_synth", "delta_accuracy", "overall_quality_score"]:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
    else:               df[c] = np.nan            # colonna assente: le sezioni che la usano lo segnaleranno

df["dataset_label"] = df["dataset"].map(DATASET_LABELS).fillna(df["dataset"])
df["method_label"]  = df["method"].map(METHOD_LABELS).fillna(df["method"])

METHODS       = sorted(df["method"].unique())
DATASET_ORDER = df.drop_duplicates("dataset").sort_values("n_features")["dataset"].tolist()   # dal più semplice al più complesso

# Una riga per (dataset, metodo)
agg = (df.groupby(["dataset", "method"], as_index=False)
         .agg(n_runs=("replica", "count"),
              n_hash=("synthetic_sha256", "nunique"), n_config=("config_sha256", "nunique"),
              n_rows=("n_rows", "first"), n_features=("n_features", "first"),
              fit_s=("fit_seconds", "mean"), fit_std=("fit_seconds", "std"),
              sample_s=("sample_seconds", "mean"), elapsed_s=("elapsed_seconds", "mean"),
              acc_synth=("accuracy_synth", "mean"), delta=("delta_accuracy", "mean"),
              quality=("overall_quality_score", "mean")))
agg["dataset_label"] = agg["dataset"].map(DATASET_LABELS).fillna(agg["dataset"])
agg["method_label"]  = agg["method"].map(METHOD_LABELS).fillna(agg["method"])
# esito riproducibilità: "untestable" = 1 sola run; "ok" = tutti gli hash uguali; "fail" = hash diversi
agg["status"]  = np.select([agg["n_runs"] < 2, agg["n_hash"] == 1], ["untestable", "ok"], default="fail")
agg["dataset"] = pd.Categorical(agg["dataset"], DATASET_ORDER, ordered=True)
agg = agg.sort_values(["dataset", "method"]).reset_index(drop=True)

print(f"{len(df)} run  ·  {df['dataset'].nunique()} dataset  ·  {len(METHODS)} metodi")

---
## 2 · I dati
**Cosa mostra.** Quante run esistono per ogni combinazione dataset × metodo. Serve a capire dove la
riproducibilità è *verificabile*: con **una sola run** non c'è nulla da confrontare, servono almeno 2 ripetizioni.

In [ ]:
runs_pivot = (agg.pivot(index="dataset_label", columns="method_label", values="n_runs")
                 .reindex([DATASET_LABELS.get(d, d) for d in DATASET_ORDER]).fillna(0).astype(int))
runs_pivot.index.name, runs_pivot.columns.name = "Dataset", None

(runs_pivot.style
    .background_gradient(cmap="Blues", vmin=0, vmax=max(3, runs_pivot.to_numpy().max()))
    .format("{:d}")
    .set_caption("Numero di run per dataset e metodo"))

In [ ]:
n_test = int((agg["n_runs"] > 1).sum())
if n_test == 0:
    display(Markdown("> ⚠️ **Nessuna combinazione ha più di una run.** La riproducibilità non è verificabile: per farlo, "
                     "aggiungi ripetizioni a seed fisso (`repeats_for_reproducibility` in `experiment_grid.yaml`). "
                     "Le sezioni successive che dipendono dalle ripetizioni lo segnalano."))
else:
    no_rep = agg[agg["n_runs"] < 2]
    msg = f"> ✅ **{n_test} combinazioni su {len(agg)}** hanno almeno 2 ripetizioni e sono verificabili."
    if len(no_rep):
        msg += " **Non verificabili** (1 sola run): " + ", ".join(f"{r.dataset_label} / {r.method_label}" for r in no_rep.itertuples()) + "."
    display(Markdown(msg))

---
## 3 · Auditabilità: la traccia di ogni run è completa?
**Cosa mostra.** Una lista di controlli sull'indice delle run. Un audit trail utile risponde a quattro domande su
ogni run: *dove* sono i suoi file, *quando* è stata eseguita, *con quale configurazione*, *che output ha prodotto*.
Gli hash SHA-256 servono a rendere le ultime due verificabili nel tempo.

In [ ]:
REQUIRED = ["run_dir", "timestamp", "config_sha256", "synthetic_sha256"]
present  = [c for c in REQUIRED if c in df.columns]
n        = len(df)

def _exists(p):
    return isinstance(p, str) and (ROOT / p).is_dir()
def _has_report(p):
    return isinstance(p, str) and (ROOT / p).is_dir() and any((ROOT / p).rglob(PHASE_REPORT))

sha_cols = [c for c in ["config_sha256", "synthetic_sha256"] if c in df.columns]
checks = [
    ("Campi obbligatori presenti e non nulli", int(df[present].notna().all(axis=1).sum()) if len(present) == len(REQUIRED) else 0),
    ("Hash SHA-256 ben formati (64 caratteri esadecimali)", int(df[sha_cols].apply(lambda s: s.astype(str).str.fullmatch(r"[0-9a-f]{64}")).all(axis=1).sum()) if sha_cols else 0),
    ("La cartella della run esiste su disco", int(df["run_dir"].apply(_exists).sum()) if "run_dir" in df.columns else 0),
    (f"Il file {PHASE_REPORT} è presente nella run", int(df["run_dir"].apply(_has_report).sum()) if "run_dir" in df.columns else 0),
]
audit = pd.DataFrame([{"Controllo": name, "Run conformi": f"{k} / {n}", "Esito": "✅" if k == n else ("⚠️" if k > 0 else "❌")}
                      for name, k in checks]).set_index("Controllo")
display(audit)

missing_cols = [c for c in REQUIRED if c not in df.columns]
if missing_cols:
    display(Markdown("> ❌ **Colonne mancanti dall'indice:** " + ", ".join(f"`{c}`" for c in missing_cols)))
if checks[2][1] < n:
    display(Markdown("> ℹ️ Se le cartelle non vengono trovate ma esistono, controlla che `run_dir` sia relativo alla **root del progetto** "
                     f"(`{ROOT}`)."))

### 3.1 L'hash della configurazione identifica la configurazione?
`config_sha256` dovrebbe essere una *firma della configurazione*: due run con la **stessa** configurazione e lo stesso
seed dovrebbero avere lo **stesso** `config_sha256`. Se invece cambia anche tra ripetizioni identiche, allora
identifica la run (probabilmente perché include l'indice di replica o il percorso) e non serve a confrontare configurazioni.

In [ ]:
same_out = agg[(agg["n_runs"] > 1) & (agg["status"] == "ok")]          # ripetizioni con output identico
diff_cfg = same_out[same_out["n_config"] > 1]

if same_out.empty:
    display(Markdown("> ℹ️ Nessuna ripetizione con output identico su cui fare il confronto."))
elif len(diff_cfg):
    lst = ", ".join(f"{r.dataset_label} / {r.method_label} ({r.n_config} hash di config su {r.n_runs} run)" for r in diff_cfg.itertuples())
    display(Markdown("> ⚠️ **`config_sha256` cambia tra ripetizioni che hanno prodotto lo stesso output:** " + lst + ". "
                     "Molto probabilmente l'hash include un campo specifico della run (indice di replica, `run_dir`, timestamp). "
                     "Per l'auditabilità conviene avere **due** hash distinti: uno della configurazione *sperimentale* (uguale tra ripetizioni) "
                     "e uno dell'identità della run."))
else:
    display(Markdown("> ✅ Le ripetizioni con output identico hanno anche lo stesso `config_sha256`: l'hash identifica la configurazione."))

---
## 4 · Riproducibilità
**Cosa mostra.** Ogni riga è una combinazione dataset × metodo; ogni **quadrato** è una ripetizione, con dentro i
primi caratteri dell'hash SHA-256 del dataset sintetico che ha prodotto.

* **Stesso colore = stesso hash** → le ripetizioni hanno prodotto un dataset identico, bit per bit
* **Colori diversi** → dataset sintetici diversi a parità di seed: la pipeline non è riproducibile
* **Quadrato vuoto** → una sola run: niente da confrontare

**Cosa significa.** Un hash uguale è una prova forte (identità bit a bit). Un hash *diverso* non implica però una
differenza statistica rilevante: per questo, nella sezione 4.2, si guarda anche di quanto cambiano le metriche.

In [ ]:
HASH_COLORS = ["#264653", "#E9C46A", "#8E6C8A", "#457B9D", "#F4A261", "#6D9F71"]
STATUS = {"ok":         ("✓ riproducibile", C_OK),
          "fail":       ("✗ non riproducibile", C_FAIL),
          "untestable": ("— non verificabile", C_MUTED)}

max_rep = int(agg["n_runs"].max())
fig, ax = plt.subplots(figsize=(10, 0.66 * len(agg) + 2.0))

for i, r in enumerate(agg.itertuples()):
    if i > 0 and agg.loc[i - 1, "dataset"] != r.dataset:            # linea di separazione tra dataset
        ax.axhline(i - 0.5, color="#E8ECEF", lw=1.2, zorder=0)
    runs = df[(df["dataset"] == r.dataset) & (df["method"] == r.method)].sort_values("replica")
    uniq = list(dict.fromkeys(runs["synthetic_sha256"]))              # ordine di prima apparizione
    for k, run in enumerate(runs.itertuples()):
        idx = uniq.index(run.synthetic_sha256)
        if r.n_runs < 2: fc, ec, tc = "white", "#CBD2D9", C_MUTED
        else:
            fc, ec = HASH_COLORS[idx % len(HASH_COLORS)], "white"
            tc = C_TEXT if idx % len(HASH_COLORS) == 1 else "white"
        ax.add_patch(Rectangle((k + 0.05, i - 0.34), 0.9, 0.68, fc=fc, ec=ec, lw=1.5, zorder=3))
        ax.text(k + 0.5, i, str(run.synthetic_sha256)[:6], ha="center", va="center", fontsize=8.5,
                family="monospace", color=tc, zorder=4)
    label, col = STATUS[r.status]
    detail = (f"{r.n_hash} hash distint{'o' if r.n_hash == 1 else 'i'} su {r.n_runs} run") if r.n_runs > 1 else "1 sola run"
    ax.text(max_rep + 0.3, i - 0.12, label, va="center", ha="left", fontsize=10.5, fontweight="bold", color=col)
    ax.text(max_rep + 0.3, i + 0.20, detail, va="center", ha="left", fontsize=9, color=C_MUTED)

ax.set_xlim(0, max_rep + 3.4); ax.set_ylim(len(agg) - 0.5, -0.6)
ax.set_yticks(range(len(agg)))
ax.set_yticklabels([f"{r.dataset_label}  ·  {r.method_label}" for r in agg.itertuples()], color=C_TEXT)
ax.set_xticks([k + 0.5 for k in range(max_rep)]); ax.set_xticklabels([f"ripetiz. {k + 1}" for k in range(max_rep)], fontsize=9)
ax.xaxis.tick_top(); ax.tick_params(length=0)
ax.grid(False)
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_title("Hash del dataset sintetico per ripetizione", pad=30)
fig.text(0.0, -0.01, "Stesso colore = stesso hash SHA-256  ·  colori diversi = output diversi a parità di seed", fontsize=9, color=C_MUTED)
finish(fig, "01_riproducibilita_hash")

In [ ]:
# Riepilogo automatico dell'esito
grp = {k: agg[agg["status"] == k] for k in ("ok", "fail", "untestable")}
names = lambda g: ", ".join(f"**{r.dataset_label} / {r.method_label}**" for r in g.itertuples())

lines = []
if len(grp["ok"]):   lines.append(f"* ✅ **Riproducibili** ({len(grp['ok'])}): {names(grp['ok'])}. Tutte le ripetizioni hanno prodotto lo stesso identico dataset.")
if len(grp["fail"]): lines.append(f"* ❌ **Non riproducibili** ({len(grp['fail'])}): {names(grp['fail'])}. A parità di seed l'output cambia.")
if len(grp["untestable"]): lines.append(f"* ➖ **Non verificabili** ({len(grp['untestable'])}, una sola run): {names(grp['untestable'])}.")
display(Markdown("**Esito**\n\n" + "\n".join(lines)))

### 4.2 Quanto cambiano le metriche tra ripetizioni?
Se l'hash è identico le metriche devono coincidere (differenza 0). Se l'hash cambia, questa tabella dice se la
differenza è trascurabile o sostanziale. Il tempo di fit varia sempre un po' (rumore di macchina) e non è un indizio di non-riproducibilità.

In [ ]:
rows = []
for r in agg[agg["n_runs"] > 1].itertuples():
    g = df[(df["dataset"] == r.dataset) & (df["method"] == r.method)]
    rng = lambda c: g[c].max() - g[c].min()
    rows.append({"Dataset": r.dataset_label, "Metodo": r.method_label, "Ripetizioni": r.n_runs,
                 "Hash distinti": r.n_hash, "Δ max−min accuracy sintetico": rng("accuracy_synth"),
                 "Δ max−min fedeltà": rng("overall_quality_score"), "Δ max−min fit (s)": rng("fit_seconds")})

if rows:
    var_df = pd.DataFrame(rows).set_index(["Dataset", "Metodo"])
    display(var_df.style.format({"Δ max−min accuracy sintetico": "{:.4f}", "Δ max−min fedeltà": "{:.4f}", "Δ max−min fit (s)": "{:.1f}"}, na_rep="n/d")
                  .set_caption("Variazione tra ripetizioni (0.0000 = identico)"))
else:
    display(Markdown("> ℹ️ Nessuna combinazione con ripetizioni: niente da confrontare."))

---
## 5 · Costo computazionale
### 5.1 Quanto tempo richiede il sintetizzatore?
**Cosa mostra.** A sinistra, il tempo di addestramento (`fit`) per dataset e metodo, in **scala logaritmica**: ogni
gradino verticale è un fattore 10. I puntini bianchi sono le singole ripetizioni (se ce ne sono più di una).
A destra, come il tempo **cresce con il numero di righe**.

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8), sharey=True, gridspec_kw={"width_ratios": [1.25, 1]})

# ── sinistra: barre per dataset e metodo ───────────────────────────────────
n = len(METHODS); width = 0.8 / n
vals = agg["fit_s"].dropna()
for j, m in enumerate(METHODS):
    for i, ds in enumerate(DATASET_ORDER):
        row = agg[(agg["dataset"] == ds) & (agg["method"] == m)]
        if row.empty or pd.isna(row["fit_s"].iloc[0]): continue
        x, v = i + (j - (n - 1) / 2) * width, row["fit_s"].iloc[0]
        ax.bar(x, v, width * 0.92, color=color_of(m), zorder=3)
        pts = df[(df["dataset"] == ds) & (df["method"] == m)]["fit_seconds"].dropna().to_numpy()
        if len(pts) > 1:
            ax.scatter(x + np.linspace(-0.18, 0.18, len(pts)) * width, pts, s=26, color="white", edgecolor=C_TEXT, linewidth=0.9, zorder=5)
        top = max(v, pts.max()) if len(pts) else v            # l'etichetta sta sopra barra e puntini
        ax.text(x, top * 1.3, fmt_time(v), ha="center", va="bottom", fontsize=10, fontweight="bold", color=color_of(m))

d1 = agg.drop_duplicates("dataset").set_index("dataset")
ax.set_xticks(range(len(DATASET_ORDER)))
ax.set_xticklabels([f"{d1.loc[ds, 'dataset_label']}\n{fmt_int(d1.loc[ds, 'n_rows'])} righe" for ds in DATASET_ORDER], color=C_TEXT)
ax.tick_params(axis="x", length=0); ax.grid(axis="x", visible=False)
ax.set_title("Tempo di fit per dataset")

# ── destra: scaling con il numero di righe ─────────────────────────────────
for m in METHODS:
    s = agg[agg["method"] == m].sort_values("n_rows")
    ax2.plot(s["n_rows"], s["fit_s"], color=color_of(m), marker="o", ms=9, mec="white", mew=1.6, lw=1.8, zorder=3)
ax2.set_xscale("log")
_d = agg.drop_duplicates("dataset").sort_values("n_rows")
ax2.set_xticks(_d["n_rows"]); ax2.set_xticklabels([f"{r.dataset_label.split()[0]}\n{fmt_int(r.n_rows)}" for r in _d.itertuples()], fontsize=9)
ax2.xaxis.set_minor_formatter(NullFormatter()); ax2.tick_params(which="minor", length=0)
ax2.set_xlabel("Numero di righe (scala log)"); ax2.set_title("Al crescere dei dati")
ax2.margins(x=0.12)

# ── asse y comune, in scala log con tick leggibili ─────────────────────────
ax.set_yscale("log")
lo, hi = 10 ** np.floor(np.log10(vals.min())), vals.max() * 4
ax.set_ylim(lo, hi)
ticks = [(t, l) for t, l in [(1, "1 s"), (10, "10 s"), (60, "1 min"), (600, "10 min"), (3600, "1 h"), (36000, "10 h")] if lo <= t <= hi]
ax.set_yticks([t for t, _ in ticks]); ax.set_yticklabels([l for _, l in ticks]); ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_ylabel("Tempo di fit (scala log)")

ax2.legend(handles=[Line2D([], [], marker="s", ls="", color=color_of(m), ms=10, label=label_of(m)) for m in METHODS] +
                   [Line2D([], [], marker="o", ls="", mfc="white", mec=C_TEXT, ms=6, label="singola ripetizione (barre a sinistra)")],
           loc="upper left", fontsize=9.5)
fig.tight_layout()
finish(fig, "02_costo_fit")

In [ ]:
# Commento automatico: rapporto di costo di ogni metodo rispetto al metodo di riferimento
if BASE_METHOD in METHODS and len(METHODS) > 1:
    piv = agg.pivot(index="dataset", columns="method", values="fit_s")
    out = []
    for m in [x for x in METHODS if x != BASE_METHOD]:
        ratio = (piv[m] / piv[BASE_METHOD]).dropna()
        if len(ratio):
            out.append(f"* **{label_of(m)}** richiede da **{ratio.min():.1f}×** a **{ratio.max():.1f}×** il tempo di fit di "
                       f"{label_of(BASE_METHOD)} (massimo su {DATASET_LABELS.get(ratio.idxmax(), ratio.idxmax())}: "
                       f"{fmt_time(piv.loc[ratio.idxmax(), m])} contro {fmt_time(piv.loc[ratio.idxmax(), BASE_METHOD])}).")
    display(Markdown("**Cosa notare**\n\n" + "\n".join(out)))
else:
    display(Markdown(f"> ℹ️ Serve il metodo di riferimento `{BASE_METHOD}` e almeno un altro metodo per calcolare i rapporti di costo."))

### 5.2 Dove va il tempo di una run?
**Cosa mostra.** La durata totale di una run (`elapsed_seconds`) divisa in: **addestramento** del sintetizzatore,
**campionamento** e **tutto il resto** (ingestione, valutazione di fedeltà/privacy, addestramento dei modelli di utility,
scrittura dell'audit trail). Il numero a destra è il tempo totale.

**Perché è interessante.** Un metodo può essere velocissimo a generare i dati e comunque non velocizzare la
pipeline, se il grosso del tempo va altrove.

In [ ]:
if agg["elapsed_s"].notna().any():
    p = agg.copy()
    p["other_s"] = (p["elapsed_s"] - p["fit_s"] - p["sample_s"]).clip(lower=0)
    p["total_s"] = p["fit_s"] + p["sample_s"] + p["other_s"]
    PARTS = [("fit_s", "Addestramento sintetizzatore", "#264653", "white"),
             ("sample_s", "Campionamento", "#E9C46A", C_TEXT),
             ("other_s", "Resto della pipeline", "#D5DBE1", C_TEXT)]

    fig, ax = plt.subplots(figsize=(9.5, 0.62 * len(p) + 1.9))
    for i, r in enumerate(p.itertuples()):
        left = 0.0
        for col, _, color, tcol in PARTS:
            w = getattr(r, col) / r.total_s
            ax.barh(i, w, left=left, color=color, height=0.64, edgecolor="white", linewidth=1.5, zorder=3)
            if w > 0.07: ax.text(left + w / 2, i, f"{w:.0%}", ha="center", va="center", fontsize=9.5, fontweight="bold", color=tcol, zorder=4)
            left += w
        ax.text(1.015, i, f"totale {fmt_time(r.total_s)}", va="center", fontsize=9.5, color=C_MUTED)

    ax.set_ylim(len(p) - 0.5, -0.5); ax.set_xlim(0, 1)
    ax.set_yticks(range(len(p))); ax.set_yticklabels([f"{r.dataset_label}  ·  {r.method_label}" for r in p.itertuples()], color=C_TEXT)
    ax.tick_params(axis="y", length=0); ax.grid(axis="y", visible=False); ax.spines["left"].set_visible(False)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.set_xlabel("Quota del tempo totale della run")
    ax.set_title("Dove va il tempo di una run")
    ax.legend(handles=[Patch(color=c, label=l) for _, l, c, _ in PARTS], loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=9.5)
    finish(fig, "03_ripartizione_tempo")

    share = (p["fit_s"] / p["total_s"]).rename("q")
    hi_, lo_ = p.loc[share.idxmax()], p.loc[share.idxmin()]
    display(Markdown(f"**Cosa notare.** Il sintetizzatore pesa dal **{share.min():.0%}** ({lo_['dataset_label']} / {lo_['method_label']}) "
                     f"al **{share.max():.0%}** ({hi_['dataset_label']} / {hi_['method_label']}) del tempo totale di una run: "
                     "il resto è valutazione e training dei modelli."))
else:
    display(Markdown("> ℹ️ `elapsed_seconds` non è disponibile: impossibile calcolare la ripartizione del tempo."))

### 5.3 Dettaglio per fase (opzionale)
L'indice tidy contiene solo il tempo totale. Il tempo **per fase** (`phase_timings`) sta nel file
`phase1_quality_report.json` dentro la cartella di ogni run. Se i file ci sono, qui compare la ripartizione fine;
altrimenti la sezione lo dice.

In [ ]:
def read_phase_timings(run_dir):
    # Legge phase_timings dal report della run. Ritorna {fase: secondi} oppure None.
    if not isinstance(run_dir, str): return None
    base = ROOT / run_dir
    try:
        candidates = [base / PHASE_REPORT, *base.rglob(PHASE_REPORT)] if base.is_dir() else []
        for p in candidates:
            if not p.exists(): continue
            t = json.loads(p.read_text(encoding="utf-8")).get("phase_timings")
            if isinstance(t, dict):
                out = {}
                for k, v in t.items():
                    if isinstance(v, dict): v = v.get("seconds", v.get("elapsed_seconds"))
                    if isinstance(v, (int, float)): out[k] = float(v)
                if out: return out
    except (OSError, ValueError):
        pass
    return None

recs = []
for r in df.itertuples():
    t = read_phase_timings(r.run_dir)
    if t: recs.append({"dataset": r.dataset, "method": r.method, **t})

if not recs:
    display(Markdown(f"> ℹ️ **Nessun `phase_timings` trovato** (cercato `{PHASE_REPORT}` nelle cartelle `run_dir`). "
                     "Quando i report per-run sono presenti, qui compare la ripartizione per fase. "
                     "Per ora vale la ripartizione a tre voci della sezione 5.2."))
else:
    ph = pd.DataFrame(recs).groupby(["dataset", "method"]).mean(numeric_only=True)
    ph = ph.reindex([(d, m) for d in DATASET_ORDER for m in METHODS if (d, m) in ph.index])
    PH_COLORS = ["#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51", "#8E6C8A", "#457B9D", "#6D9F71"]
    fig, ax = plt.subplots(figsize=(9.5, 0.62 * len(ph) + 1.9))
    for i, ((ds, m), row) in enumerate(ph.iterrows()):
        tot, left = row.sum(), 0.0
        for k, (phase, v) in enumerate(row.dropna().items()):
            w = v / tot
            ax.barh(i, w, left=left, color=PH_COLORS[k % len(PH_COLORS)], height=0.64, edgecolor="white", linewidth=1.5, zorder=3)
            if w > 0.07: ax.text(left + w / 2, i, f"{w:.0%}", ha="center", va="center", fontsize=9, fontweight="bold",
                                 color=C_TEXT if k % len(PH_COLORS) in (2, 3) else "white", zorder=4)
            left += w
        ax.text(1.015, i, f"totale {fmt_time(tot)}", va="center", fontsize=9.5, color=C_MUTED)
    ax.set_ylim(len(ph) - 0.5, -0.5); ax.set_xlim(0, 1)
    ax.set_yticks(range(len(ph))); ax.set_yticklabels([f"{DATASET_LABELS.get(d, d)}  ·  {label_of(m)}" for d, m in ph.index], color=C_TEXT)
    ax.tick_params(axis="y", length=0); ax.grid(axis="y", visible=False); ax.spines["left"].set_visible(False)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.set_xlabel("Quota del tempo per fase")
    ax.set_title("Tempo per fase della pipeline")
    ax.legend(handles=[Patch(color=PH_COLORS[k % len(PH_COLORS)], label=c) for k, c in enumerate(ph.columns)],
              loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=min(4, len(ph.columns)), fontsize=9)
    finish(fig, "04_tempo_per_fase")

---
## 6 · Costo vs beneficio
**Cosa mostra.** Per ogni dataset, quanto costa in più un metodo rispetto al riferimento (asse x, in scala log:
×10 = dieci volte più lento) e quanto guadagna in **utility** (asse y, punti percentuali di accuracy in più).
L'utility viene dalla RQ1 (Δ accuracy sintetico − reale).

**Come leggerlo.**
* **Alto a destra** → costa di più *e* rende di più: il costo va valutato caso per caso
* **Basso a destra** → costa di più *e* rende meno: il metodo costoso non conviene
* Il metodo di riferimento è quello a costo ×1, a sinistra del grafico

In [ ]:
cb = pd.DataFrame()
if BASE_METHOD in METHODS and len(METHODS) > 1:
    base = agg[agg["method"] == BASE_METHOD].set_index("dataset")
    rows = []
    for r in agg[agg["method"] != BASE_METHOD].itertuples():
        if r.dataset not in base.index: continue
        b = base.loc[r.dataset]
        if pd.isna(r.fit_s) or pd.isna(b["fit_s"]) or pd.isna(r.delta) or pd.isna(b["delta"]): continue
        rows.append({"dataset_label": r.dataset_label, "method": r.method, "method_label": r.method_label,
                     "cost_x": r.fit_s / b["fit_s"], "util_pp": (r.delta - b["delta"]) * 100,
                     "quality_gain": r.quality - b["quality"] if pd.notna(r.quality) and pd.notna(b["quality"]) else np.nan})
    cb = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7.4, 5.0))
if cb.empty:
    ax.text(0.5, 0.5, "Confronto non disponibile\nservono il metodo di riferimento e almeno\nun altro metodo con fit time e utility",
            transform=ax.transAxes, ha="center", va="center", fontsize=11, color=C_MUTED, linespacing=1.6,
            bbox=dict(boxstyle="round,pad=0.9", fc="#F3F4F6", ec="#D1D5DB", ls="--"))
    ax.set_title("Costo extra vs utility guadagnata")
else:
    for m in cb["method"].unique():
        s = cb[cb["method"] == m]
        ax.scatter(s["cost_x"], s["util_pp"], s=170, color=color_of(m), edgecolor="white", linewidth=1.8, zorder=3)
        for r in s.itertuples():
            ax.annotate(r.dataset_label.split()[0], (r.cost_x, r.util_pp), xytext=(10, 8), textcoords="offset points", fontsize=9.5, color=C_MUTED)
    ax.axhline(0, color=C_TEXT, lw=1, alpha=0.6, ls=(0, (4, 3)), zorder=0)
    ax.axvline(1, color=C_TEXT, lw=1, alpha=0.35, ls=":", zorder=0)
    ax.set_xscale("log")
    ax.set_xlim(min(0.8, cb["cost_x"].min() / 1.5), cb["cost_x"].max() * 2.2)
    ymax = max(abs(cb["util_pp"]).max() * 1.35, 3)
    ax.set_ylim(-ymax, ymax)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"×{v:g}")); ax.xaxis.set_minor_formatter(NullFormatter())
    ax.text(0.98, 0.97, "costa di più, rende di più", transform=ax.transAxes, ha="right", va="top", fontsize=9, color=C_MUTED, style="italic")
    ax.text(0.98, 0.03, "costa di più, rende meno", transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color=C_MUTED, style="italic")
    ax.set_xlabel(f"Costo di fit rispetto a {label_of(BASE_METHOD)} (scala log)")
    ax.set_ylabel(f"Utility guadagnata rispetto a {label_of(BASE_METHOD)} (pp di accuracy)")
    ax.set_title("Il costo extra si ripaga in utility?")
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=color_of(m), ms=10, mec="white", label=label_of(m)) for m in cb["method"].unique()],
              loc="upper left", fontsize=9.5)
finish(fig, "05_costo_vs_beneficio")

if not cb.empty:
    txt = [f"* **{r.dataset_label}** · {r.method_label}: costa **×{r.cost_x:.1f}** in tempo di fit e rende **{fmt_pp(r.util_pp)}** di accuracy "
           f"({'più' if r.util_pp > 0 else 'meno'} utility) rispetto a {label_of(BASE_METHOD)}." for r in cb.itertuples()]
    display(Markdown("**Cosa notare**\n\n" + "\n".join(txt)))

---
## 7 · Tabella di sintesi
Riproducibilità e costo per ogni dataset e metodo, pronti da citare.

In [ ]:
STATUS_TXT = {"ok": "✓ riproducibile", "fail": "✗ non riproducibile", "untestable": "— non verificabile"}
summary_df = pd.DataFrame({
    "Dataset": agg["dataset_label"], "Metodo": agg["method_label"], "Run": agg["n_runs"],
    "Hash distinti": agg["n_hash"], "Riproducibilità": agg["status"].map(STATUS_TXT),
    "Fit (s)": agg["fit_s"], "Fit std (s)": agg["fit_std"], "Sample (s)": agg["sample_s"], "Totale run (s)": agg["elapsed_s"],
    "Quota sintetizzatore (%)": (agg["fit_s"] + agg["sample_s"]) / agg["elapsed_s"] * 100,
}).set_index(["Dataset", "Metodo"])

def _color_status(v):
    return f"color: {C_OK}; font-weight: 600" if v.startswith("✓") else (f"color: {C_FAIL}; font-weight: 600" if v.startswith("✗") else f"color: {C_MUTED}")

sty = (summary_df.style
    .format({"Fit (s)": "{:.1f}", "Fit std (s)": "{:.1f}", "Sample (s)": "{:.2f}", "Totale run (s)": "{:.0f}", "Quota sintetizzatore (%)": "{:.0f}"}, na_rep="—")
    .bar(subset=["Quota sintetizzatore (%)"], color="#A8DADC", vmin=0, vmax=100)
    .set_caption("Riproducibilità e costo computazionale (media sulle ripetizioni)"))
(sty.map if hasattr(sty, "map") else sty.applymap)(_color_status, subset=["Riproducibilità"])   # .map da pandas 2.1, prima applymap

**Export per la tesi** — la stessa tabella in LaTeX (richiede `booktabs`):

In [ ]:
latex_df = summary_df.copy()
latex_df["Riproducibilità"] = latex_df["Riproducibilità"].str.replace("✓ ", "", regex=False).str.replace("✗ ", "", regex=False).str.replace("— ", "", regex=False)
print(latex_df.round(2).to_latex(float_format="%.2f", na_rep="--", multicolumn_format="c",
                                 caption="Riproducibilità e costo computazionale per dataset e metodo.", label="tab:rq3-summary"))

---
## 8 · Limiti e prossimi passi
Cosa questa analisi può e non può concludere. Buono da riportare nella sezione *Limitazioni* della tesi.

| Limite | Perché conta | Come affrontarlo |
|---|---|---|
| **La riproducibilità è verificata solo su una macchina** | Un hash identico prova l'identità bit a bit *su questo ambiente*: versione di Python/PyTorch, CPU o GPU e numero di thread possono cambiarlo | Registrare nell'audit trail versioni delle librerie e hardware, e ripetere su una seconda macchina |
| **Metodi senza ripetizioni** (una sola run) | La riproducibilità non è verificabile, non è «dimostrata». In particolare, il determinismo atteso di Gaussian Copula va confermato, non assunto | Aggiungere `repeats_for_reproducibility` anche per i metodi che ne sono privi |
| **L'hash è sul file, non sui dati** | L'SHA-256 dipende anche da formattazione, ordine delle colonne e precisione dei float: un hash diverso non implica dati statisticamente diversi | Confrontare anche le metriche (sezione 4.2) o un hash calcolato sul DataFrame normalizzato |
| **`config_sha256` può identificare la run e non la configurazione** | Se cambia tra ripetizioni identiche non permette di confrontare configurazioni (vedi 3.1) | Separare l'hash della configurazione sperimentale da quello dell'identità della run |
| **Tempi misurati su poche run e su una macchina non dedicata** | Warm-up, altri processi e cache introducono rumore | Riportare hardware, e usare più ripetizioni se serve una stima robusta |
| **Solo il tempo come costo** | Memoria, energia e costo monetario non sono misurati | Aggiungere picco di RAM e, se possibile, consumo energetico |
| **Il costo/beneficio usa l'utility della RQ1** | Eredita i suoi limiti (pochi dataset, un solo classificatore) | Estendere ai nuovi dataset man mano |

**Idee per estendere il notebook**: una verifica automatica dell'audit trail in `aggregate.py` (fallire se manca un
campo), un confronto tra ambienti diversi, e il consumo di memoria per fase.